In [1]:
import requests

import numpy as np
import pandas as pd

import plotly.express as px
import folium

import os
from pathlib import Path
from urllib.request import urlretrieve
from urllib.error import HTTPError, URLError
from zipfile import ZipFile

In [2]:
#Data Enrichment
#1.Read the concatinated data
#2.Handle Missing values
#3.Create derived columns for further analysis
#4.Save the data in data/JC

In [3]:
#Read the concatinated data
citibike_df = pd.read_csv("../data/citibike/JC2025.csv")
citibike_df = pd.read_csv("../data/citibike/JC2026.csv")

#### Step 7: Setting Dates


In [4]:
citibike_df['started_at'] = pd.to_datetime(citibike_df['started_at'], errors='coerce')
citibike_df['ended_at'] = pd.to_datetime(citibike_df['ended_at'], errors='coerce')

In [5]:
#check data types
print(citibike_df[['started_at', 'ended_at']].dtypes)

started_at    datetime64[us]
ended_at      datetime64[us]
dtype: object


#### Step 8: df overview


In [6]:
citibike_df.info

<bound method DataFrame.info of                  ride_id  rideable_type              started_at  \
0       40FBC7AEEBB220EC  electric_bike 2026-01-11 15:01:01.954   
1       47FB1CA65D3F7E0F  electric_bike 2026-01-16 11:49:41.364   
2       BFB10DE1E40F3709  electric_bike 2026-01-07 19:55:41.424   
3       F16B896495AAF618  electric_bike 2026-01-24 11:12:10.547   
4       76D77D5F1D491B3C  electric_bike 2026-01-22 20:58:00.602   
...                  ...            ...                     ...   
415721  DDCC16C0541F379B  electric_bike 2026-06-17 00:08:40.697   
415722  63E5BDCB0E6B66F8   classic_bike 2026-06-24 18:26:29.824   
415723  08693F34F752A9CE  electric_bike 2026-06-17 00:01:18.305   
415724  6914EA3CD9787D17   classic_bike 2026-06-07 12:51:01.894   
415725  524A4D471D0AB1FC  electric_bike 2026-06-02 17:58:09.236   

                      ended_at                 start_station_name  \
0      2026-01-11 15:16:57.103  14 St Ferry - 14 St & Shipyard Ln   
1      2026-01-16 11:58:2

In [7]:
citibike_df.columns

Index(['ride_id', 'rideable_type', 'started_at', 'ended_at',
       'start_station_name', 'start_station_id', 'end_station_name',
       'end_station_id', 'start_lat', 'start_lng', 'end_lat', 'end_lng',
       'member_casual', 'year'],
      dtype='str')

#### Step 9. Handling the missig values

In [8]:
citibike_df.shape

(415726, 14)

In [9]:
missing_values = (
    citibike_df
    .isna()
    .sum()
    .reset_index()
)

missing_values.columns = ["column", "missing_count"]

missing_values["missing_share"] = (
    missing_values["missing_count"] / len(citibike_df)
)

missing_values.sort_values("missing_count", ascending=False)

,column,missing_count,missing_share
7,end_station_id,2856,0.006870
10,end_lat,2842,0.006836
11,end_lng,2842,0.006836
6,end_station_name,2105,0.005063
4,start_station_name,1,0.000002
5,start_station_id,1,0.000002
8,start_lat,1,0.000002
9,start_lng,1,0.000002
0,ride_id,0,0.000000
1,rideable_type,0,0.000000


#### Step 10: Ride Duration


In [10]:
#Now we create ride duration.
citibike_df["ride_duration_minutes"] = (
    citibike_df["ended_at"] - citibike_df["started_at"]
).dt.total_seconds() / 60

##### Removing extremes

In [11]:
citibike_df = citibike_df.dropna(
    subset=[
        "ride_id",
        "started_at",
        "ended_at",
        "start_lat",
        "start_lng",
        "end_lat",
        "end_lng"
    ]
)


In [12]:
citibike_df = citibike_df[
    (citibike_df["ride_duration_minutes"] > 1) &
    (citibike_df["ride_duration_minutes"] <= 24 * 60)
].copy()

#### Step 11: Time Based Variables


In [13]:
#For later aggregation, we create useful time columns.
citibike_df["date"] = citibike_df["started_at"].dt.date
citibike_df["month"] = citibike_df["started_at"].dt.to_period("M").astype(str)
citibike_df["month_name"] = citibike_df["started_at"].dt.month_name()
citibike_df["day_of_week"] = citibike_df["started_at"].dt.day_name()
citibike_df["hour"] = citibike_df["started_at"].dt.hour

In [14]:
citibike_df.head()

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual,year,ride_duration_minutes,date,month,month_name,day_of_week,hour
0,40FBC7AEEBB220EC,electric_bike,2026-01-11 15:01:01.954,2026-01-11 15:16:57.103,14 St Ferry - 14 St & Shipyard Ln,HB202,Pier 61 at Chelsea Piers,6233.04,40.752961,-74.024353,40.746872,-74.00821,member,2026,15.919150,2026-01-11,2026-01,January,Sunday,15
1,47FB1CA65D3F7E0F,electric_bike,2026-01-16 11:49:41.364,2026-01-16 11:58:27.291,14 St Ferry - 14 St & Shipyard Ln,HB202,Madison St & 1 St,HB402,40.752961,-74.024353,40.738790,-74.03930,member,2026,8.765450,2026-01-16,2026-01,January,Friday,11
2,BFB10DE1E40F3709,electric_bike,2026-01-07 19:55:41.424,2026-01-07 19:59:09.874,Church Sq Park - 5 St & Park Ave,HB601,Madison St & 1 St,HB402,40.742659,-74.032233,40.738790,-74.03930,member,2026,3.474167,2026-01-07,2026-01,January,Wednesday,19
3,F16B896495AAF618,electric_bike,2026-01-24 11:12:10.547,2026-01-24 11:34:42.801,Madison St & 1 St,HB402,Madison St & 1 St,HB402,40.738790,-74.039300,40.738790,-74.03930,member,2026,22.537567,2026-01-24,2026-01,January,Saturday,11
4,76D77D5F1D491B3C,electric_bike,2026-01-22 20:58:00.602,2026-01-22 21:02:39.088,Clinton St & 7 St,HB303,Madison St & 1 St,HB402,40.745420,-74.033320,40.738790,-74.03930,casual,2026,4.641433,2026-01-22,2026-01,January,Thursday,20


In [15]:
def assign_season(month_number):
    if month_number in [12, 1, 2]:
        return "Winter"
    elif month_number in [3, 4, 5]:
        return "Spring"
    elif month_number in [6, 7, 8]:
        return "Summer"
    else:
        return "Autumn"


citibike_df["season"] = (
    citibike_df["started_at"]
    .dt.month
    .apply(assign_season)
)

In [16]:
citibike_df[
    [
        "started_at",
        "date",
        "month",
        "month_name",
        "day_of_week",
        "hour",
        "season"
    ]
].head()

,started_at,date,month,month_name,day_of_week,hour,season
0,2026-01-11 15:01:01.954,2026-01-11,2026-01,January,Sunday,15,Winter
1,2026-01-16 11:49:41.364,2026-01-16,2026-01,January,Friday,11,Winter
2,2026-01-07 19:55:41.424,2026-01-07,2026-01,January,Wednesday,19,Winter
3,2026-01-24 11:12:10.547,2026-01-24,2026-01,January,Saturday,11,Winter
4,2026-01-22 20:58:00.602,2026-01-22,2026-01,January,Thursday,20,Winter


In [17]:
# Split data by year

citibike_2025 = citibike_df[citibike_df["year"] == 2025]

citibike_2026 = citibike_df[citibike_df["year"] == 2026]

#### Store the data

In [18]:
#Once we have created all the required derived columns we can save enriched yearly datasets as separate CSV files

citibike_2025.to_csv(
    "../data/citibike/JC/JC2025_Enriched.csv",
    index=False
)

citibike_2026.to_csv(
    "../data/citibike/JC/JC2026_Enriched.csv",
    index=False
)